# 02 - Structured Streaming + Auto Loader (hand-written)

**Prerequisite:** run `00_Setup_Mock_Data` first and use the same **catalog** / **schema** widget values here.

What this notebook shows:
1. **Auto Loader** (`cloudFiles`) incremental file ingestion into a bronze Delta table, with schema inference, schema hints and rescued data
2. **Stateful streaming**: event-time windowed aggregation with a **watermark** (handles late events)
3. **Append output mode**: windows are written once the watermark finalizes them (no duplicates, no MERGE needed)
4. **Stream-static join** + **`foreachBatch`** custom sink: enrich orders with a product dimension and route bad rows to a quarantine table
5. **Incremental behaviour**: drop a new file, re-run, only the new file is processed

`trigger(availableNow=True)` processes everything available and then stops - ideal for scheduled jobs (and the only trigger type on serverless). On classic compute you can use `processingTime` for always-on streams.

**Trade-offs vs. SDP**: here *you* manage checkpoints, schema locations, query ordering, DQ, retries and table creation. In the SDP pipeline (`pipeline/`) the same ingestion is ~5 lines and those concerns are handled by the engine.

In [0]:
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "de_demo", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
spark.sql("USE CATALOG IDENTIFIER(:c)", args={"c": CATALOG})
spark.sql("USE SCHEMA IDENTIFIER(:s)", args={"s": SCHEMA})

# Keep checkpoints & schema tracking OUTSIDE the landing folder Auto Loader reads from
# (the `checkpoints` volume is created by 00_Setup_Mock_Data)
CKPT = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/ss_demo"
print(LANDING, CKPT)

## 1. Auto Loader -> bronze
* `cloudFiles.schemaLocation`: where Auto Loader stores the inferred schema and tracks evolution
* `cloudFiles.inferColumnTypes` + `schemaHints`: real types instead of all-strings
* `_rescued_data`: captures values that don't match the schema instead of dropping them
* `_metadata`: file path / modification time for lineage

In [0]:
from pyspark.sql import functions as F

rows_before = spark.table("ss_events_bronze").count() if spark.catalog.tableExists("ss_events_bronze") else 0

bronze_q = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CKPT}/schema/events_bronze")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "event_time TIMESTAMP, user_id BIGINT")
    .load(f"{LANDING}/events")
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingested_at", F.current_timestamp())
  .writeStream
    .option("checkpointLocation", f"{CKPT}/events_bronze")
    .trigger(availableNow=True)
    .toTable("ss_events_bronze"))

bronze_q.awaitTermination()
rows_after = spark.table("ss_events_bronze").count()
print(f"{rows_after - rows_before} new rows ingested in this run (table total: {rows_after})")
display(spark.sql("SELECT source_file, count(*) AS rows FROM ss_events_bronze GROUP BY ALL ORDER BY 1"))

## 2. Event-time windows + watermark (append mode)
* `withWatermark("event_time", "30 minutes")`: events up to 30 min late are still counted; older state is dropped (bounded memory)
* `outputMode("append")`: a window is written **once**, when the watermark passes its end, so the table never needs updates. The most recent windows stay in state until newer data advances the watermark: land a new events file (section 4) and re-run to see them appear.
* The mock data contains ~3% late events (10-90 min late); events later than the watermark are dropped.

Alternative: `outputMode("update")` + `foreachBatch` with a MERGE upserts windows as they change. While building this demo on serverless, MERGE inside `foreachBatch` made the session unusable, so this notebook uses append mode. For upserts, see the MERGE in notebook 01 and AUTO CDC in the SDP pipeline.

In [0]:
agg_q = (spark.readStream.table("ss_events_bronze")
    .withWatermark("event_time", "30 minutes")
    .groupBy(F.window("event_time", "5 minutes"), "event_type")
    .agg(F.count("*").alias("events"), F.approx_count_distinct("user_id").alias("users"))
    .select(F.col("window.start").alias("window_start"), F.col("window.end").alias("window_end"),
            "event_type", "events", "users")
  .writeStream
    .outputMode("append")          # emit each window once, after the watermark closes it
    .option("checkpointLocation", f"{CKPT}/event_windows")
    .trigger(availableNow=True)
    .toTable("ss_event_windows"))
agg_q.awaitTermination()

display(spark.sql("""
  SELECT count(*) AS finalized_windows, sum(events) AS events, max(window_end) AS latest_window_end
  FROM ss_event_windows"""))
display(spark.sql("SELECT * FROM ss_event_windows ORDER BY window_start, event_type LIMIT 20"))

## 3. Stream-static join + filtering bad records
Orders are streamed with Auto Loader and enriched with the (static) product dimension. Invalid rows are routed to a quarantine table instead of silently dropped, done by hand with `foreachBatch` writing to two sinks. In SDP this is a one-line `@dp.expect_or_drop`.

In [0]:
products = (spark.read.format("csv").option("header", True).option("inferSchema", True)
            .load(f"{LANDING}/products"))

def route_orders(batch_df, batch_id):
    valid = F.col("customer_id").isNotNull() & (F.col("amount") >= 0)
    batch_df.filter(valid).write.mode("append").saveAsTable("ss_orders_enriched")
    batch_df.filter(~valid | F.col("customer_id").isNull()).write.mode("append").saveAsTable("ss_orders_quarantine")

orders_q = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CKPT}/schema/orders")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "order_ts TIMESTAMP")
    .load(f"{LANDING}/orders")
    .join(products, "product_id", "left")          # stream-static join
  .writeStream
    .foreachBatch(route_orders)
    .option("checkpointLocation", f"{CKPT}/orders")
    .trigger(availableNow=True)
    .start())
orders_q.awaitTermination()

display(spark.sql("""
  SELECT 'enriched' AS tbl, count(*) AS rows FROM ss_orders_enriched
  UNION ALL SELECT 'quarantine', count(*) FROM ss_orders_quarantine"""))

## 4. Show incremental processing
Run the next cell to land a new events file, then **re-run the Auto Loader cell** above: only the new file (2,000 rows) is ingested, thanks to the checkpoint.

In [0]:
import json, random, uuid
from datetime import datetime, timedelta

batch = uuid.uuid4().hex[:6]
t0 = datetime(2026, 11, 1, 6)
rows = [json.dumps({"event_id": f"N{batch}-{k}", "user_id": random.randint(1, 200),
                    "event_type": random.choice(["page_view", "add_to_cart", "checkout", "search"]),
                    "page": random.choice(["/home", "/product", "/cart", "/search"]),
                    "device": random.choice(["web", "ios", "android"]),
                    "event_time": (t0 + timedelta(seconds=random.randint(0, 3599))).isoformat()})
        for k in range(2000)]
path = f"{LANDING}/events/events_new_{batch}.json"
dbutils.fs.put(path, "\n".join(rows), overwrite=False)
print("landed", path)

## 5. Observability
`lastProgress` / `recentProgress` expose per-micro-batch metrics (durations, watermark, state size). In production push these via a `StreamingQueryListener`; SDP gives you an event log and pipeline UI out of the box.

In [0]:
# Progress objects expose metrics as attributes (dict-style .get() returns None on serverless).
# Note: on serverless, row counters in progress can read 0 - count rows in the target table for exact numbers.
for name, q in [("bronze", bronze_q), ("event_windows", agg_q)]:
    for p in q.recentProgress:
        print(f"{name}: batch={p.batchId} duration_ms={p.batchDuration} watermark={p.eventTime.get('watermark')} "
              f"state_rows={[s.numRowsTotal for s in p.stateOperators]}")

## Reset the demo (optional)
Uncomment and run to drop the tables and checkpoints so the demo can be replayed from scratch.

In [0]:
# for t in ["ss_events_bronze", "ss_event_windows", "ss_orders_enriched", "ss_orders_quarantine"]:
#     spark.sql("DROP TABLE IF EXISTS IDENTIFIER(:t)", args={"t": t})
# dbutils.fs.rm(CKPT, recurse=True)